**1. Install Libraries**

In [5]:
!pip install -q --upgrade langchain langchain-core langchain-community transformers accelerate faiss-cpu

**2**. LLM Setup (HuggingFace - FREE)

In [6]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="google/flan-t5-base",   # MUCH better than distilgpt2
    max_new_tokens=100
)

llm = HuggingFacePipeline(pipeline=pipe)

print(llm.invoke("What is LangChain?"))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

What is LangChain?


**3. Prompt Template**

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer clearly and briefly:\nQuestion: {question}"
)

**4. Chain (LCEL - Latest Method)**

In [8]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"question": "What are LangChain components?"}))

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer clearly and briefly:
Question: What are LangChain components?


**5. Memory (Custom Implementation)**

In [9]:
chat_history = []

memory_prompt = PromptTemplate(
    input_variables=["history", "question"],
    template="""
Answer the question based on conversation.

Conversation:
{history}

Question: {question}
Answer:
"""
)

memory_chain = memory_prompt | llm | StrOutputParser()

def chat(question):
    global chat_history

    history_text = "\n".join(chat_history[-4:])  # LIMIT history (IMPORTANT)

    response = memory_chain.invoke({
        "history": history_text,
        "question": question
    })

    chat_history.append(f"User: {question}")
    chat_history.append(f"AI: {response}")

    return response

# Test
print(chat("Hi, I am Srinidhi"))
print(chat("What is my name?"))

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question based on conversation.

Conversation:


Question: Hi, I am Srinidhi
Answer:



Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer the question based on conversation.

Conversation:
User: Hi, I am Srinidhi
AI: 
Answer the question based on conversation.

Conversation:


Question: Hi, I am Srinidhi
Answer:


Question: What is my name?
Answer:



**6. Vector Store (FAISS + Embeddings)**

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document   # ✅ FIXED

# Documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="It supports chains, memory, and agents.")
]

# Embeddings
embeddings = HuggingFaceEmbeddings()

# Vector store
vectorstore = FAISS.from_documents(docs, embeddings)

# Query
results = vectorstore.similarity_search("What is LangChain?")

# Output
for doc in results:
    print(doc.page_content)

/tmp/ipykernel_5677/1234639570.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_5677/1234639570.py:12: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

LangChain helps build LLM applications.
It supports chains, memory, and agents.
